# Bronze layer


In [0]:
table_name = 'flightdata.bronze.flightData'
source_data = '/Volumes/flightdata/landing-data/landing-vol/flights.csv'
source_format = 'CSV'

# Drop the existing (incorrectly-schema'd) table so it can be recreated properly
spark.sql("DROP TABLE IF EXISTS " + table_name)

spark.sql("CREATE TABLE IF NOT EXISTS " + table_name)

spark.sql("COPY INTO " + table_name + \
  " FROM '" + source_data + "'" + \
  " FILEFORMAT = " + source_format + \
  " FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true', 'mergeSchema' = 'true', 'multiLine' = 'true')" + \
  " COPY_OPTIONS ('mergeSchema' = 'true')"
)

# Verify
display(spark.sql("SELECT * FROM " + table_name + " LIMIT 5"))
spark.sql("DESCRIBE " + table_name).show(50, truncate=False)


In [0]:
%sql
select * from flightdata.bronze.flightdata


## rename and make capital each word.

In [0]:
df = spark.read.table("flightdata.bronze.flightData")
df_renamed = df.toDF(*["_".join(w.capitalize() for w in c.split("_")) for c in df.columns])
display(df_renamed)

## change all time format

In [0]:
from pyspark.sql import functions as F


def to_12hr_format(col_name):
    padded = F.lpad(F.col(col_name).cast("int").cast("string"), 4, "0")
    hour = F.substring(padded, 1, 2).cast("int")
    minute = F.substring(padded, 3, 2).cast("int")

    hour12 = F.when(hour == 0, 12).when(hour > 12, hour - 12).otherwise(hour)
    period = F.when(hour < 12, "AM").otherwise("PM")

    return F.when(F.col(col_name).isNull(), None).otherwise(
        F.concat(
            hour12.cast("string"),
            F.lit(":"),
            F.lpad(minute.cast("string"), 2, "0"),
            F.lit(" "),
            period,
        )
    )


time_columns = ["Dep_Time", "Sched_Dep_Time", "Arr_Time", "Sched_Arr_Time"]

# Apply all transformations at once using withColumns (more efficient than loop)
df_clean = df_renamed.withColumns({col_name: to_12hr_format(col_name) for col_name in time_columns})

display(df_clean.select(*time_columns))

## cleanup time , distance column,

In [0]:
from pyspark.sql import functions as F

df_clean = df

# 1. Air_Time -> readable duration "3h 47m"
df_clean = df_clean.withColumn(
    "Air_Time",
    F.when(F.col("Air_Time").isNull(), None).otherwise(
        F.concat(
            (F.col("Air_Time") / 60).cast("int").cast("string"), F.lit("h "),
            (F.col("Air_Time") % 60).cast("int").cast("string"), F.lit("m")
        )
    )
)

# 2. Distance -> rename to Distance_Miles, add Distance_Km
df_clean = df_clean.withColumnRenamed("Distance", "Distance_Miles")
df_clean = df_clean.withColumn(
    "Distance_Km",
    F.round(F.col("Distance_Miles") * 1.60934, 1)
)

# 3. Time_Hour -> simplified formatting, drop timezone/milliseconds
df_clean = df_clean.withColumn(
    "Time_Hour",
    F.date_format(F.col("Time_Hour"), "yyyy-MM-dd HH:mm")
)

display(df_clean.select("Air_Time", "Distance_Miles", "Distance_Km", "Time_Hour"))

In [0]:
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("flightdata.bronze.flightData")

display(spark.read.table("flightdata.bronze.flightData"))